# 01 — WSI to Tissue Sections

This notebook walks through the core pipeline: loading flat whole-slide images (WSI),
segmenting individual tissue sections, extracting them as tiles, and renaming with
global slice indices.

**Prerequisites:** `pip install -e .` from the `wsi-tissue-pipeline` root.

**Pipeline stages covered:**
1. Configure paths and segmentation parameters
2. Run `process_directory()` to segment + extract tiles
3. Quick visual check of output tiles
4. Manual artifact removal (user-driven)
5. Rename files with global slice indices via `rename_outputs_by_overall_index()`

In [ ]:
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
from PIL import Image

from wsi_pipeline.config import PipelineConfig, SegmentationConfig, TileConfig, OutputConfig
from wsi_pipeline.wsi_processing import process_directory
from wsi_pipeline.tiles.naming import rename_outputs_by_overall_index

%load_ext autoreload
%autoreload 2

## Step 1: Configure Paths and Parameters

Set the input WSI directory, file pattern, and output directory.  Then create a
`PipelineConfig` with segmentation parameters tuned for your data.

Key `SegmentationConfig` fields:
- **`backend`** — segmentation algorithm (`"local-entropy"`, `"otsu"`, etc.)
- **`target_long_side`** — thumbnail long-side in px for mask computation
- **`min_area_px`** — discard connected components smaller than this
- **`struct_elem_px`** — morphological closing disk radius
- **`split_touching`** / **`r_split`** — separate fused tissue sections

In [ ]:
# --- Edit these paths for your data ---
subject_wsi_dir = Path(
    "./data/input"  # ← Replace with the path to your WSI flat files
).expanduser().resolve()

pattern = "*.jpg"  # ← Adjust glob pattern to match your WSI filenames

output_dir = Path(
    "./data/output"  # ← Replace with your desired output directory
).expanduser().resolve()

# --- Pipeline configuration ---
config = PipelineConfig(
    segmentation=SegmentationConfig(
        backend="local-entropy",
        target_long_side=1800,
        min_area_px=3000,
        struct_elem_px=4,
        split_touching=False,
        r_split=2,
        diagnostics=True,
    ),
    tiles=TileConfig(
        chunk_size=512,
        pad_multiple=512,
        extra_margin_px=0,
    ),
    output=OutputConfig(
        format="tiff",
        convert_to_uint8=False,
    ),
    specimen_spacing=9,  # e.g., macaque: every 10th slice collected
)

print(config.model_dump_json(indent=2))

## Step 2: Process the WSI Directory

`process_directory()` iterates over all files matching `pattern`, segments tissue
regions, extracts individual tiles, and writes them to `output_dir`.

In [ ]:
# Clean output directory for a fresh run
if output_dir.exists():
    shutil.rmtree(output_dir)
output_dir.mkdir(parents=True, exist_ok=True)

# Run the processing pipeline
results = process_directory(
    subject_wsi_dir,
    output_dir,
    pattern=pattern,
    config=config,
)

# Summary
total_tiles = sum(len(v) for v in results.values())
print(f"\nProcessed {len(results)} WSI files -> {total_tiles} tissue tiles")

In [ ]:
# List output tiles
tiles = sorted(output_dir.glob("*.tif"))
print(f"{len(tiles)} output tiles")
for t in tiles[:10]:
    print(f"  {t.name}")
if len(tiles) > 10:
    print(f"  ... and {len(tiles) - 10} more")

## Step 3: Quick Visual Check

Display a few sample tiles to verify segmentation quality.  For a comprehensive
QC review with contact sheets, see **notebook 02**.

In [ ]:
import random

sample = random.sample(tiles, min(4, len(tiles)))
fig, axes = plt.subplots(1, len(sample), figsize=(4 * len(sample), 4))
if len(sample) == 1:
    axes = [axes]
for ax, path in zip(axes, sample):
    ax.imshow(Image.open(path))
    ax.set_title(path.name, fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()

## Step 4: Manual Artifact Removal

After inspecting the QC contact sheet (notebook 02), identify non-tissue tiles
(glass, debris, etc.) and delete them here.  Then re-run the renaming step below.

For example, in the macaque dataset tile `E241_right_level_7_Image_11_01_0351`
was a piece of glass that needed to be removed.

In [ ]:
# Example: delete a problematic tile
# Replace the filename with the actual tile to remove
fname_to_delete = output_dir / "EXAMPLE_NAME.tif"
if fname_to_delete.exists():
    fname_to_delete.unlink()
    print(f"Deleted: {fname_to_delete.name}")
else:
    print(f"Not found (expected): {fname_to_delete.name}")

## Step 5: Rename Files with Global Indices

`rename_outputs_by_overall_index()` assigns a `_ZZZZ` suffix to each tile using
the arithmetic progression `start + rank * (spacing + 1)`.  For the macaque
dataset with every 10th slice collected, `spacing=9` produces indices
`0001, 0011, 0021, ...`

In [ ]:
renames = rename_outputs_by_overall_index(
    output_dir,
    pattern=pattern,
    spacing=9,
    pad=4,
    start=1,
    dry_run=False,
)

print(f"Renamed {len(renames)} files")
for old, new in renames[:5]:
    print(f"  {old.name} -> {new.name}")
if len(renames) > 5:
    print("  ...")
    for old, new in renames[-3:]:
        print(f"  {old.name} -> {new.name}")

## Step 6 (Optional): Process Additional Pyramid Levels

Different pyramid levels may need different segmentation parameters.  Uncomment
and adjust the cell below for a second level.

In [ ]:
# # --- Level 06 example ---
# subject_wsi_dir_06 = Path(
#     "./data/input_level06"  # ← Replace with your level 06 flat files
# ).expanduser().resolve()
# pattern_06 = "*.jpg"
# output_dir_06 = Path(
#     "./data/output_level06"  # ← Replace with your level 06 output directory
# ).expanduser().resolve()
#
# config_06 = PipelineConfig(
#     segmentation=SegmentationConfig(
#         backend="local-entropy",
#         target_long_side=2400,
#         min_area_px=6000,
#         struct_elem_px=7,
#         split_touching=False,
#         r_split=2,
#         diagnostics=True,
#     ),
#     tiles=TileConfig(chunk_size=512, pad_multiple=512, extra_margin_px=0),
#     output=OutputConfig(format="tiff", convert_to_uint8=False),
#     specimen_spacing=9,
# )
#
# if output_dir_06.exists():
#     shutil.rmtree(output_dir_06)
# output_dir_06.mkdir(parents=True, exist_ok=True)
#
# results_06 = process_directory(
#     subject_wsi_dir_06, output_dir_06, pattern=pattern_06, config=config_06
# )

## Next Steps

- **Notebook 02** — Generate full QC contact sheets to identify artifacts
- **Notebook 03** — Visualize extracted tissues in Neuroglancer
- **Notebook 04** — Prepare the image stack for EM-LDDMM registration